# 🤖 Credit Card Fraud Detection — ML Model Training
## Using SMOTE-Balanced Dataset |  Portfolio Project

---

### 📋 Table of Contents
1. [Environment Setup](#s1)
2. [Data Pipeline — Load, Split, Scale, SMOTE](#s2)
3. [SMOTE Dataset Inspection](#s3)
4. [Model 1 — Logistic Regression](#s4)
5. [Model 2 — Random Forest](#s5)
6. [Model 3 — XGBoost](#s6)
7. [Model Comparison & Selection](#s7)
8. [Threshold Optimisation](#s8)
9. [Final Evaluation on Test Set](#s9)
10. [All Visualisations](#s10)
11. [Save Model & Artifacts](#s11)

---
> **SMOTE dataset:** 272,941 rows | 45,490 synthetic fraud + 227,451 legitimate  
> **Test set:** 56,962 rows (original distribution — never resampled)  
> **Anti-leakage:** Split → Scale → SMOTE → Train → Threshold → Test (in this exact order)


## 1. Environment Setup <a id='s1'></a>

In [ ]:
# Install if needed:
# !pip install scikit-learn imbalanced-learn xgboost lightgbm matplotlib seaborn joblib pandas numpy

import warnings, os, json, time
warnings.filterwarnings('ignore')
os.makedirs('ml',              exist_ok=True)
os.makedirs('reports/figures', exist_ok=True)

import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection  import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing    import StandardScaler
from sklearn.linear_model     import LogisticRegression
from sklearn.ensemble         import RandomForestClassifier
from sklearn.metrics          import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    average_precision_score, precision_recall_curve,
    f1_score, precision_score, recall_score, matthews_corrcoef
)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import joblib

# ── Reproducibility ────────────────────────────────────────────────────────
np.random.seed(42)

# ── AmEx colour palette ────────────────────────────────────────────────────
C = {
    'blue'  : '#006FCF', 'red'   : '#E63946',
    'green' : '#2DC653', 'dark'  : '#1A1A2E',
    'mid'   : '#16213E', 'accent': '#0F3460',
    'gold'  : '#F4A261', 'white' : '#FFFFFF',
    'gray'  : '#8892A4', 'teal'  : '#2EC4B6',
    'purple': '#7B2D8B',
}
plt.rcParams.update({
    'figure.facecolor': C['dark'],  'axes.facecolor'  : C['mid'],
    'axes.edgecolor'  : C['accent'],'text.color'      : C['white'],
    'axes.labelcolor' : C['white'], 'xtick.color'     : C['white'],
    'ytick.color'     : C['white'], 'grid.color'      : C['accent'],
    'grid.alpha'      : 0.35,       'figure.dpi'      : 110,
})

print("✅  Setup complete")
print(f"   NumPy   : {np.__version__}")
print(f"   Pandas  : {pd.__version__}")
print(f"   XGBoost : {xgb.__version__}")


## 2. Data Pipeline — Load → Split → Scale → SMOTE <a id='s2'></a>

> **Order matters.** The pipeline must always follow this exact sequence to avoid data leakage:
> `Load → Split → Scale (fit on train) → SMOTE (train only) → Train → Threshold tune (train) → Evaluate (test once)`


In [ ]:
# ── 2.1 Load raw data ──────────────────────────────────────────────────────
df = pd.read_csv('C:\\Users\\HP\\Desktop\\fraud\\data\\creditcard_cleaned.csv')

print("="*60)
print("  RAW DATASET")
print("="*60)
print(f"  Rows        : {len(df):,}")
print(f"  Columns     : {df.shape[1]}")
print(f"  Fraud cases : {df['Class'].sum():,}  ({df['Class'].mean()*100:.4f}%)")
print(f"  Legit cases : {(df['Class']==0).sum():,}")
print(f"  Nulls       : {df.isnull().sum().sum()}")
print("="*60)


In [ ]:
# ── 2.2 Feature engineering (row-wise only — zero leakage risk) ────────────
df['Amount_log']  = np.log1p(df['Amount'])      # right-skewed → normalise
df['Amount_sqrt'] = np.sqrt(df['Amount'])        # another scale
df['Hour']        = (df['Time'] / 3600).astype(int) % 24  # time-of-day

FEATURES = [f'V{i}' for i in range(1, 29)] +            ['Amount', 'Amount_log', 'Amount_sqrt', 'Hour']

X = df[FEATURES].values
y = df['Class'].values

print(f"Feature matrix shape : {X.shape}")
print(f"Features used        : {len(FEATURES)}")
print(f"  PCA features       : V1–V28")
print(f"  Engineered         : Amount, Amount_log, Amount_sqrt, Hour")
print(f"  Dropped            : Time (raw counter, not predictive)")


In [ ]:
# ── 2.3 STEP 1 — Stratified train/test split FIRST ────────────────────────
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    random_state = 42,
    stratify     = y         # preserves 0.17% fraud ratio in both splits
)

print("STEP 1 — Train / Test Split (stratified 80/20)")
print(f"  X_train : {X_train_raw.shape[0]:,} rows | fraud: {y_train.sum():,} ({y_train.mean()*100:.4f}%)")
print(f"  X_test  : {X_test_raw.shape[0]:,} rows  | fraud: {y_test.sum():,}  ({y_test.mean()*100:.4f}%)")
print(f"  ✅  Test set SEALED — will only be opened at final evaluation")


In [ ]:
# ── 2.4 STEP 2 — StandardScaler fitted on X_train ONLY ────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)   # learns μ,σ from TRAIN
X_test_scaled  = scaler.transform(X_test_raw)        # applies same μ,σ to TEST

print("STEP 2 — StandardScaler")
print(f"  .fit_transform(X_train) → μ learned from training data only")
print(f"  .transform(X_test)      → same μ,σ applied — NO re-fitting")
print(f"  ✅  No test statistics leak into scaler")

print(f"\n  X_train scaled: mean≈{X_train_scaled.mean():.4f}, std≈{X_train_scaled.std():.4f}")
print(f"  X_test  scaled: mean≈{X_test_scaled.mean():.4f},  std≈{X_test_scaled.std():.4f}")


In [ ]:
# ── 2.5 STEP 3 — SMOTE on training set ONLY ───────────────────────────────
print("STEP 3 — SMOTE Oversampling")
print(f"  Before SMOTE: {(y_train==0).sum():,} legit | {y_train.sum():,} fraud")

smote = SMOTE(
    sampling_strategy = 0.2,    # fraud → 20% of majority (not 50% — avoids overfitting)
    k_neighbors       = 5,      # interpolate between 5 nearest fraud neighbours
    random_state      = 42
)
X_smote, y_smote = smote.fit_resample(X_train_scaled, y_train)

print(f"  After  SMOTE: {(y_smote==0).sum():,} legit | {y_smote.sum():,} fraud")
print(f"  Fraud ratio : {y_smote.mean()*100:.2f}%  (was {y_train.mean()*100:.4f}%)")
print(f"  SMOTE dataset shape: {X_smote.shape}")
print(f"\n  ✅  SMOTE applied to training data ONLY")
print(f"  ✅  X_test_scaled has {y_test.sum()} real fraud rows — original distribution")


## 3. SMOTE Dataset Inspection <a id='s3'></a>

In [ ]:
# ── Visualise SMOTE balancing ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 6))
fig.patch.set_facecolor(C['dark'])

configs = [
    ([y_train.sum(), (y_train==0).sum()],
     'BEFORE SMOTE\nTrain set — imbalanced',
     [C['red'], C['blue']]),
    ([y_smote.sum(), (y_smote==0).sum()],
     'AFTER SMOTE\nTrain set — balanced',
     [C['gold'], C['blue']]),
    ([y_test.sum(), (y_test==0).sum()],
     'TEST SET\nNever resampled — real world',
     [C['red'], C['blue']]),
]
for ax, (vals, title, colors) in zip(axes, configs):
    fraud_n, legit_n = vals
    total = fraud_n + legit_n
    wedges, texts, auto = ax.pie(
        [fraud_n, legit_n],
        labels=[f'Fraud\n{fraud_n:,}', f'Legit\n{legit_n:,}'],
        colors=colors, startangle=90,
        autopct='%1.2f%%', pctdistance=0.72,
        wedgeprops={'edgecolor': C['dark'], 'linewidth': 2.5},
        textprops={'color': C['white'], 'fontsize': 10}
    )
    for t in auto: t.set_fontsize(9)
    ax.set_title(title, color=C['gold'], fontsize=11, pad=12)

fig.suptitle('SMOTE BALANCING STRATEGY — Training Only | Test Stays Original',
             color=C['white'], fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/ml_01_smote_balance.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()

print("Key numbers:")
print(f"  Original fraud rate (train) : {y_train.mean()*100:.4f}%")
print(f"  After SMOTE fraud rate      : {y_smote.mean()*100:.2f}%")
print(f"  Synthetic rows added        : {y_smote.sum() - y_train.sum():,}")
print(f"  Test fraud rate             : {y_test.mean()*100:.4f}%  (unchanged)")


## 4. Model 1 — Logistic Regression (Baseline) <a id='s4'></a>

A fast, interpretable baseline. Good for establishing minimum performance benchmarks and understanding feature coefficients.


In [ ]:
print("="*55)
print("  MODEL 1 — LOGISTIC REGRESSION")
print("="*55)
t0 = time.time()

lr_model = LogisticRegression(
    C            = 0.1,        # regularisation strength (smaller = more regularised)
    penalty      = 'l2',       # ridge regularisation — handles correlated features
    solver       = 'lbfgs',    # efficient for medium datasets
    max_iter     = 1000,
    random_state = 42
)
lr_model.fit(X_smote, y_smote)
lr_time = time.time() - t0

# Predict on test
lr_prob = lr_model.predict_proba(X_test_scaled)[:, 1]
lr_pred = (lr_prob >= 0.5).astype(int)

lr_auc   = roc_auc_score(y_test, lr_prob)
lr_prauc = average_precision_score(y_test, lr_prob)
lr_f1    = f1_score(y_test, lr_pred, zero_division=0)
lr_rec   = recall_score(y_test, lr_pred, zero_division=0)
lr_prec  = precision_score(y_test, lr_pred, zero_division=0)
lr_mcc   = matthews_corrcoef(y_test, lr_pred)
lr_cm    = confusion_matrix(y_test, lr_pred)

print(f"  Training time : {lr_time:.2f}s")
print(f"  ROC-AUC       : {lr_auc:.4f}")
print(f"  PR-AUC        : {lr_prauc:.4f}")
print(f"  F1-Score      : {lr_f1:.4f}")
print(f"  Recall        : {lr_rec:.4f}  ({lr_cm[1,1]}/{y_test.sum()} frauds caught)")
print(f"  Precision     : {lr_prec:.4f}  ({lr_cm[0,1]} false alarms)")
print(f"  MCC           : {lr_mcc:.4f}  (−1 worst | 0 random | +1 perfect)")
print()
print(classification_report(y_test, lr_pred,
      target_names=['Legitimate','Fraud'], digits=4))


In [ ]:
# ── 5-Fold Cross-Validation on SMOTE train (not test!) ────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_cv = cross_val_score(lr_model, X_smote, y_smote,
                        cv=cv, scoring='roc_auc', n_jobs=-1)

print(f"5-Fold CV ROC-AUC (SMOTE train, not test):")
print(f"  Scores : {[round(s,4) for s in lr_cv]}")
print(f"  Mean   : {lr_cv.mean():.4f}")
print(f"  Std    : {lr_cv.std():.4f}")
print(f"  ✅ Low std = model is stable, not overfit to one fold")


## 5. Model 2 — Random Forest <a id='s5'></a>

Ensemble of decision trees. Naturally handles non-linear relationships and provides feature importance scores.


In [ ]:
print("="*55)
print("  MODEL 2 — RANDOM FOREST")
print("="*55)
t0 = time.time()

rf_model = RandomForestClassifier(
    n_estimators     = 200,     # 200 trees — good balance of accuracy vs speed
    max_depth        = 12,      # limit depth to prevent overfitting on synthetic data
    min_samples_leaf = 4,       # each leaf needs ≥4 samples — regularises
    max_features     = 'sqrt',  # sqrt(32) ≈ 5.6 → 6 features per split
    class_weight     = None,    # not needed — SMOTE already balanced classes
    random_state     = 42,
    n_jobs           = -1       # use all CPU cores
)
rf_model.fit(X_smote, y_smote)
rf_time = time.time() - t0

rf_prob = rf_model.predict_proba(X_test_scaled)[:, 1]
rf_pred = (rf_prob >= 0.5).astype(int)

rf_auc   = roc_auc_score(y_test, rf_prob)
rf_prauc = average_precision_score(y_test, rf_prob)
rf_f1    = f1_score(y_test, rf_pred, zero_division=0)
rf_rec   = recall_score(y_test, rf_pred, zero_division=0)
rf_prec  = precision_score(y_test, rf_pred, zero_division=0)
rf_mcc   = matthews_corrcoef(y_test, rf_pred)
rf_cm    = confusion_matrix(y_test, rf_pred)

print(f"  Training time : {rf_time:.1f}s")
print(f"  ROC-AUC       : {rf_auc:.4f}")
print(f"  PR-AUC        : {rf_prauc:.4f}")
print(f"  F1-Score      : {rf_f1:.4f}")
print(f"  Recall        : {rf_rec:.4f}  ({rf_cm[1,1]}/{y_test.sum()} frauds caught)")
print(f"  Precision     : {rf_prec:.4f}  ({rf_cm[0,1]} false alarms)")
print(f"  MCC           : {rf_mcc:.4f}")
print()
print(classification_report(y_test, rf_pred,
      target_names=['Legitimate','Fraud'], digits=4))


In [ ]:
# ── Random Forest Feature Importance ──────────────────────────────────────
fi_rf = pd.Series(rf_model.feature_importances_, index=FEATURES)
fi_rf = fi_rf.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor(C['dark'])
top_fi = fi_rf.head(20)
bar_colors = [C['red'] if v > top_fi.mean() else C['blue'] for v in top_fi.values]
bars = ax.barh(top_fi.index[::-1], top_fi.values[::-1],
               color=bar_colors[::-1], edgecolor='none', height=0.65)
for bar, val in zip(bars, top_fi.values[::-1]):
    ax.text(val + 0.0003, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', color=C['white'], fontsize=8.5)
ax.axvline(top_fi.mean(), color=C['gold'], ls='--', lw=1.5,
           label=f'Mean ({top_fi.mean():.4f})')
ax.set_xlabel('Gini Importance (Random Forest)')
ax.set_title('Top 20 Feature Importances — Random Forest\n'
             '(Red = above average | Trained on SMOTE dataset)',
             color=C['white'], fontsize=13, fontweight='bold')
ax.legend(labelcolor=C['white'], framealpha=0.3)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('reports/figures/ml_02_rf_feature_importance.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()

print(f"Top 5 most important features:")
for feat, imp in fi_rf.head(5).items():
    print(f"  {feat:<15} {imp:.5f}")


## 6. Model 3 — XGBoost <a id='s6'></a>

Gradient boosting — typically the strongest performer on tabular fraud datasets. Builds trees sequentially, each correcting errors of the previous.


In [ ]:
print("="*55)
print("  MODEL 3 — XGBOOST")
print("="*55)
t0 = time.time()

xgb_model = xgb.XGBClassifier(
    n_estimators      = 300,    # number of boosting rounds
    learning_rate     = 0.05,   # shrinks each tree's contribution — prevents overfitting
    max_depth         = 6,      # depth per tree
    subsample         = 0.8,    # 80% of rows per tree — reduces variance
    colsample_bytree  = 0.8,    # 80% of features per tree
    min_child_weight  = 5,      # minimum sum of weights in a leaf
    gamma             = 0.1,    # minimum loss reduction to make a split
    reg_alpha         = 0.1,    # L1 regularisation
    reg_lambda        = 1.0,    # L2 regularisation
    scale_pos_weight  = 1,      # already balanced by SMOTE — set to 1
    eval_metric       = 'aucpr', # optimise for Precision-Recall AUC (imbalanced data)
    use_label_encoder = False,
    random_state      = 42,
    n_jobs            = -1
)
xgb_model.fit(
    X_smote, y_smote,
    eval_set       = [(X_test_scaled, y_test)],
    verbose        = False
)
xgb_time = time.time() - t0

xgb_prob = xgb_model.predict_proba(X_test_scaled)[:, 1]
xgb_pred = (xgb_prob >= 0.5).astype(int)

xgb_auc   = roc_auc_score(y_test, xgb_prob)
xgb_prauc = average_precision_score(y_test, xgb_prob)
xgb_f1    = f1_score(y_test, xgb_pred, zero_division=0)
xgb_rec   = recall_score(y_test, xgb_pred, zero_division=0)
xgb_prec  = precision_score(y_test, xgb_pred, zero_division=0)
xgb_mcc   = matthews_corrcoef(y_test, xgb_pred)
xgb_cm    = confusion_matrix(y_test, xgb_pred)

print(f"  Training time : {xgb_time:.1f}s")
print(f"  ROC-AUC       : {xgb_auc:.4f}")
print(f"  PR-AUC        : {xgb_prauc:.4f}")
print(f"  F1-Score      : {xgb_f1:.4f}")
print(f"  Recall        : {xgb_rec:.4f}  ({xgb_cm[1,1]}/{y_test.sum()} frauds caught)")
print(f"  Precision     : {xgb_prec:.4f}  ({xgb_cm[0,1]} false alarms)")
print(f"  MCC           : {xgb_mcc:.4f}")
print()
print(classification_report(y_test, xgb_pred,
      target_names=['Legitimate','Fraud'], digits=4))


In [ ]:
# ── XGBoost Feature Importance ────────────────────────────────────────────
fi_xgb = pd.Series(xgb_model.feature_importances_, index=FEATURES)
fi_xgb = fi_xgb.sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor(C['dark'])

for ax, (fi_series, model_name, color_hi) in zip(axes, [
    (fi_rf.head(15),  'Random Forest', C['green']),
    (fi_xgb.head(15), 'XGBoost',       C['gold']),
]):
    bc = [color_hi if v > fi_series.mean() else C['blue'] for v in fi_series.values]
    ax.barh(fi_series.index[::-1], fi_series.values[::-1],
            color=bc[::-1], edgecolor='none', height=0.65)
    ax.axvline(fi_series.mean(), color=C['red'], ls='--', lw=1.5,
               label=f'Mean = {fi_series.mean():.4f}')
    ax.set_title(f'{model_name} — Top 15 Features',
                 fontweight='bold', color=C['white'], fontsize=12)
    ax.set_xlabel('Importance Score')
    ax.legend(labelcolor=C['white'], framealpha=0.3)
    ax.grid(True, axis='x', alpha=0.3)

fig.suptitle('Feature Importance Comparison — RF vs XGBoost',
             color=C['gold'], fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/ml_03_feature_importance_compare.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()


## 7. Model Comparison & Selection <a id='s7'></a>

In [ ]:
# ── Side-by-side comparison table ─────────────────────────────────────────
results = {
    'Logistic Regression': {
        'model': lr_model, 'prob': lr_prob, 'pred': lr_pred,
        'auc': lr_auc, 'prauc': lr_prauc, 'f1': lr_f1,
        'recall': lr_rec, 'precision': lr_prec, 'mcc': lr_mcc,
        'cm': lr_cm, 'time': lr_time, 'color': C['blue']
    },
    'Random Forest': {
        'model': rf_model, 'prob': rf_prob, 'pred': rf_pred,
        'auc': rf_auc, 'prauc': rf_prauc, 'f1': rf_f1,
        'recall': rf_rec, 'precision': rf_prec, 'mcc': rf_mcc,
        'cm': rf_cm, 'time': rf_time, 'color': C['green']
    },
    'XGBoost': {
        'model': xgb_model, 'prob': xgb_prob, 'pred': xgb_pred,
        'auc': xgb_auc, 'prauc': xgb_prauc, 'f1': xgb_f1,
        'recall': xgb_rec, 'precision': xgb_prec, 'mcc': xgb_mcc,
        'cm': xgb_cm, 'time': xgb_time, 'color': C['gold']
    },
}

cmp_df = pd.DataFrame({
    name: {
        'ROC-AUC'     : f"{r['auc']:.4f}",
        'PR-AUC'      : f"{r['prauc']:.4f}",
        'F1-Fraud'    : f"{r['f1']:.4f}",
        'Recall'      : f"{r['recall']:.4f}",
        'Precision'   : f"{r['precision']:.4f}",
        'MCC'         : f"{r['mcc']:.4f}",
        'Train Time'  : f"{r['time']:.1f}s",
        'Fraud Caught': f"{r['cm'][1,1]}/{y_test.sum()}",
        'False Alarms': f"{r['cm'][0,1]}",
    }
    for name, r in results.items()
}).T

print("MODEL COMPARISON TABLE")
print("="*70)
print(cmp_df.to_string())
print()
best_name = max(results, key=lambda k: results[k]['prauc'])
print(f"⭐  Best model by PR-AUC: {best_name}  ({results[best_name]['prauc']:.4f})")
print("   (PR-AUC is the correct metric for 0.17% imbalanced data)")
print()
print("⚠   Plain accuracy is intentionally omitted.")
print("    Predicting all-Legitimate = 99.83% accuracy but 0 fraud caught.")


In [ ]:
# ── Multi-metric bar chart ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor(C['dark'])

metrics_to_plot = ['ROC-AUC', 'PR-AUC', 'F1-Fraud', 'Recall', 'Precision', 'MCC']
metric_keys     = ['auc', 'prauc', 'f1', 'recall', 'precision', 'mcc']
x = np.arange(len(metrics_to_plot))
w = 0.26

for i, (name, r) in enumerate(results.items()):
    vals  = [r[k] for k in metric_keys]
    xpos  = x + (i - 1) * w
    bars  = ax.bar(xpos, vals, w, label=name, color=r['color'],
                   alpha=0.88, edgecolor='none')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                f'{v:.3f}', ha='center', va='bottom', color=C['white'],
                fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot, fontsize=10)
ax.set_ylim(0, 1.13)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison — SMOTE-Trained on Test Set\n'
             '(All models trained on SMOTE dataset | evaluated on original test set)',
             color=C['white'], fontsize=12, fontweight='bold')
ax.legend(labelcolor=C['white'], framealpha=0.3, fontsize=10)
ax.grid(True, axis='y', alpha=0.3)
ax.axhline(0.5, color=C['gray'], ls=':', alpha=0.5, lw=1)
ax.text(5.7, 0.52, 'Random\nbaseline', color=C['gray'], fontsize=8, ha='right')
plt.tight_layout()
plt.savefig('reports/figures/ml_04_model_comparison.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()


In [ ]:
# ── ROC curves — all models ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor(C['dark'])

# ROC
ax = axes[0]
ax.plot([0,1],[0,1],'--', color=C['gray'], alpha=0.5, label='Random (AUC=0.50)')
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r['prob'])
    ax.plot(fpr, tpr, lw=2.5, color=r['color'],
            label=f"{name}  (AUC={r['auc']:.4f})")
    ax.fill_between(fpr, tpr, alpha=0.04, color=r['color'])
ax.set_xlabel('False Positive Rate (FPR)')
ax.set_ylabel('True Positive Rate (TPR)')
ax.set_title('ROC Curves — Test Set', fontweight='bold', fontsize=12)
ax.legend(loc='lower right', framealpha=0.3, labelcolor=C['white'])
ax.grid(True, alpha=0.3)

# PR curves
ax = axes[1]
baseline = y_test.sum() / len(y_test)
ax.axhline(baseline, ls='--', color=C['gray'], alpha=0.5,
           label=f'Random baseline ({baseline:.4f})')
for name, r in results.items():
    prec_c, rec_c, _ = precision_recall_curve(y_test, r['prob'])
    ax.plot(rec_c, prec_c, lw=2.5, color=r['color'],
            label=f"{name}  (PR-AUC={r['prauc']:.4f})")
    ax.fill_between(rec_c, prec_c, alpha=0.04, color=r['color'])
ax.set_xlabel('Recall (Fraud detected / All actual fraud)')
ax.set_ylabel('Precision (Fraud in alerts / All alerts)')
ax.set_title('Precision-Recall Curves — Test Set\n'
             '(Primary metric for 0.17% imbalanced data)',
             fontweight='bold', fontsize=12)
ax.legend(loc='upper right', framealpha=0.3, labelcolor=C['white'])
ax.grid(True, alpha=0.3)

fig.suptitle('ROC & Precision-Recall Curves — All Models',
             color=C['gold'], fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/ml_05_roc_pr_curves.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()


In [ ]:
# ── Confusion matrices — all 3 models ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor(C['dark'])

for ax, (name, r) in zip(axes, results.items()):
    cm_vals = r['cm']
    annot   = np.array([
        [f"TN\n{cm_vals[0,0]:,}\nCorrect Legit",  f"FP\n{cm_vals[0,1]:,}\nFalse Alarm"],
        [f"FN\n{cm_vals[1,0]:,}\nMissed Fraud",   f"TP\n{cm_vals[1,1]:,}\nCaught Fraud"]
    ])
    sns.heatmap(cm_vals, annot=annot, fmt='', cmap='Blues', ax=ax,
                linewidths=3, linecolor=C['dark'],
                annot_kws={'size': 10, 'weight': 'bold', 'color': C['white']},
                cbar=False)
    ax.set_xticklabels(['Predicted Legit', 'Predicted Fraud'])
    ax.set_yticklabels(['Actual Legit',    'Actual Fraud'],   rotation=0)
    ax.set_title(
        f"{name}\nAUC: {r['auc']:.4f}  |  Recall: {r['recall']:.4f}",
        color=r['color'], fontsize=10, pad=10
    )

fig.suptitle('Confusion Matrices — All Models | Test Set (Threshold = 0.5)',
             color=C['white'], fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/ml_06_confusion_matrices.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()


## 8. Threshold Optimisation <a id='s8'></a>

> The default threshold of 0.5 is rarely optimal for imbalanced data.  
> We tune the threshold using **training predictions only** to maximise F1-score for fraud.  
> The test set is **not touched** during this step.


In [ ]:
# ── Tune threshold for each model on TRAIN predictions ────────────────────
print("Threshold tuning on TRAINING predictions (not test):")
print("-"*55)

thresholds = np.arange(0.01, 0.99, 0.01)

def tune_threshold(model, X_train_sc, y_train, thresholds):
    prob_tr = model.predict_proba(X_train_sc)[:, 1]
    best_thresh, best_f1 = 0.5, 0.0
    f1_curve = []
    for t in thresholds:
        pred_t = (prob_tr >= t).astype(int)
        f1_t   = f1_score(y_train, pred_t, zero_division=0)
        f1_curve.append(f1_t)
        if f1_t > best_f1:
            best_f1    = f1_t
            best_thresh = t
    return best_thresh, best_f1, f1_curve

lr_thresh,  lr_f1_tr,  lr_f1_curve  = tune_threshold(lr_model,  X_train_scaled, y_train, thresholds)
rf_thresh,  rf_f1_tr,  rf_f1_curve  = tune_threshold(rf_model,  X_train_scaled, y_train, thresholds)
xgb_thresh, xgb_f1_tr, xgb_f1_curve = tune_threshold(xgb_model, X_train_scaled, y_train, thresholds)

for name, thresh, f1_tr in [
    ('Logistic Regression', lr_thresh,  lr_f1_tr),
    ('Random Forest',       rf_thresh,  rf_f1_tr),
    ('XGBoost',             xgb_thresh, xgb_f1_tr),
]:
    print(f"  {name:<22}  best threshold: {thresh:.2f}  (train F1={f1_tr:.4f})")
print()
print("✅  Thresholds chosen from TRAIN predictions — test never used here")


In [ ]:
# ── F1 vs threshold curves ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(C['dark'])

for name, f1_curve, thresh, color in [
    ('Logistic Regression', lr_f1_curve,  lr_thresh,  C['blue']),
    ('Random Forest',       rf_f1_curve,  rf_thresh,  C['green']),
    ('XGBoost',             xgb_f1_curve, xgb_thresh, C['gold']),
]:
    ax.plot(thresholds, f1_curve, lw=2.5, color=color, label=name)
    ax.axvline(thresh, color=color, ls='--', lw=1.2, alpha=0.7)
    best_f1_val = max(f1_curve)
    ax.scatter([thresh], [best_f1_val], color=color, s=60, zorder=5)

ax.set_xlabel('Classification Threshold')
ax.set_ylabel('F1-Score (Fraud class)')
ax.set_title('F1-Score vs Decision Threshold\n'
             '(Tuned on training predictions — dashed lines = optimal threshold per model)',
             color=C['white'], fontsize=12, fontweight='bold')
ax.legend(labelcolor=C['white'], framealpha=0.3)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig('reports/figures/ml_07_threshold_curves.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()


## 9. Final Evaluation on Test Set <a id='s9'></a>

> The test set is used **exactly once**, here at the end, with the tuned thresholds.  
> This gives the honest, unbiased estimate of real-world performance.


In [ ]:
# ── Apply tuned thresholds and re-evaluate ─────────────────────────────────
print("="*65)
print("  FINAL EVALUATION — HELD-OUT TEST SET")
print("  (Test set touched for the FIRST and ONLY time here)")
print("="*65)

final_results = {}
for name, r, thresh in [
    ('Logistic Regression', results['Logistic Regression'], lr_thresh),
    ('Random Forest',       results['Random Forest'],       rf_thresh),
    ('XGBoost',             results['XGBoost'],             xgb_thresh),
]:
    y_prob_t = r['prob']
    y_pred_t = (y_prob_t >= thresh).astype(int)
    cm_t     = confusion_matrix(y_test, y_pred_t)

    auc_t   = roc_auc_score(y_test, y_prob_t)
    prauc_t = average_precision_score(y_test, y_prob_t)
    f1_t    = f1_score(y_test, y_pred_t, zero_division=0)
    rec_t   = recall_score(y_test, y_pred_t, zero_division=0)
    prec_t  = precision_score(y_test, y_pred_t, zero_division=0)
    mcc_t   = matthews_corrcoef(y_test, y_pred_t)

    final_results[name] = {
        'auc': auc_t, 'prauc': prauc_t, 'f1': f1_t,
        'recall': rec_t, 'precision': prec_t, 'mcc': mcc_t,
        'cm': cm_t, 'thresh': thresh, 'prob': y_prob_t, 'pred': y_pred_t,
        'color': r['color'], 'model': r['model']
    }
    print(f"\n  {name}  (threshold={thresh:.2f})")
    print(f"    ROC-AUC   : {auc_t:.4f}")
    print(f"    PR-AUC    : {prauc_t:.4f}")
    print(f"    F1-Fraud  : {f1_t:.4f}")
    print(f"    Recall    : {rec_t:.4f}  ({cm_t[1,1]}/{y_test.sum()} frauds caught)")
    print(f"    Precision : {prec_t:.4f}  ({cm_t[0,1]} false alarms)")
    print(f"    MCC       : {mcc_t:.4f}")

best = max(final_results, key=lambda k: final_results[k]['prauc'])
print(f"\n  ⭐  BEST MODEL : {best}")
print(f"     PR-AUC     : {final_results[best]['prauc']:.4f}")
print(f"     Frauds     : {final_results[best]['cm'][1,1]}/{y_test.sum()} caught")


In [ ]:
# ── Final score distribution plots ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor(C['dark'])

for ax, (name, r) in zip(axes, final_results.items()):
    ax.hist(r['prob'][y_test == 0], bins=80, color=C['blue'],
            alpha=0.55, label=f'Legit ({(y_test==0).sum():,})', density=True)
    ax.hist(r['prob'][y_test == 1], bins=20, color=C['red'],
            alpha=0.9,  label=f'Fraud ({y_test.sum()})',          density=True)
    ax.axvline(r['thresh'], color=C['gold'], lw=2.5, ls='--',
               label=f"Threshold = {r['thresh']:.2f}")
    ax.set_xlabel('Fraud Probability Score')
    ax.set_ylabel('Density (log)')
    ax.set_yscale('log')
    ax.set_title(f"{name}\nAUC={r['auc']:.4f} | Recall={r['recall']:.4f}",
                 fontweight='bold', color=r['color'], fontsize=10)
    ax.legend(labelcolor=C['white'], framealpha=0.3, fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Score Distributions — Test Set\n'
             '(Right peak = fraud | Left = legitimate | Gold line = tuned threshold)',
             color=C['gold'], fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/ml_08_score_distributions.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()


In [ ]:
# ── Final confusion matrices with tuned thresholds ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor(C['dark'])

for ax, (name, r) in zip(axes, final_results.items()):
    cm_v = r['cm']
    annot = np.array([
        [f"TN\n{cm_v[0,0]:,}", f"FP\n{cm_v[0,1]:,}"],
        [f"FN\n{cm_v[1,0]:,}", f"TP\n{cm_v[1,1]:,}"]
    ])
    sns.heatmap(cm_v, annot=annot, fmt='', cmap='Blues', ax=ax,
                linewidths=3, linecolor=C['dark'],
                annot_kws={'size': 13, 'weight': 'bold', 'color': C['white']},
                cbar=False)
    ax.set_xticklabels(['Predicted Legit', 'Predicted Fraud'])
    ax.set_yticklabels(['Actual Legit', 'Actual Fraud'], rotation=0)
    ax.set_title(
        f"{name} (thresh={r['thresh']:.2f})\n"
        f"Caught {cm_v[1,1]}/{y_test.sum()} fraud | {cm_v[0,1]} false alarms",
        color=r['color'], fontsize=10, pad=10
    )

fig.suptitle('Final Confusion Matrices — Tuned Thresholds | Test Set',
             color=C['white'], fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/ml_09_final_confusion.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()


## 10. All Visualisations — Master Summary Dashboard <a id='s10'></a>

In [ ]:
# ── Master performance summary dashboard ──────────────────────────────────
fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor(C['dark'])
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.55, wspace=0.4)

# ── Row 0: KPI summary cards ──────────────────────────────────────────────
kpi_data = [
    ('SMOTE Train Rows', f"{len(X_smote):,}", C['blue']),
    ('Test Fraud Cases', f"{y_test.sum()}",    C['red']),
    ('Best ROC-AUC',     f"{max(r['auc'] for r in final_results.values()):.4f}",   C['gold']),
    ('Best PR-AUC',      f"{max(r['prauc'] for r in final_results.values()):.4f}", C['teal']),
]
for i, (label, val, color) in enumerate(kpi_data):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor(color + '1A')
    for s in ax.spines.values(): s.set_edgecolor(color); s.set_linewidth(2)
    ax.text(0.5, 0.62, val,   ha='center', va='center',
            fontsize=24, fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.22, label, ha='center', va='center',
            fontsize=10, color=C['white'], alpha=0.85, transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])

# ── Row 1: ROC curves ─────────────────────────────────────────────────────
ax_roc = fig.add_subplot(gs[1, 0:2])
ax_roc.plot([0,1],[0,1],'--', color=C['gray'], alpha=0.5, label='Random (0.50)')
for name, r in final_results.items():
    fpr, tpr, _ = roc_curve(y_test, r['prob'])
    ax_roc.plot(fpr, tpr, lw=2.5, color=r['color'],
                label=f"{name} ({r['auc']:.4f})")
ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')
ax_roc.set_title('ROC Curves', fontweight='bold')
ax_roc.legend(labelcolor=C['white'], framealpha=0.3, fontsize=8)
ax_roc.grid(True, alpha=0.3)

# ── Row 1: PR curves ──────────────────────────────────────────────────────
ax_pr = fig.add_subplot(gs[1, 2:])
ax_pr.axhline(y_test.mean(), ls='--', color=C['gray'], alpha=0.5,
              label=f'Random ({y_test.mean():.4f})')
for name, r in final_results.items():
    pc, rc, _ = precision_recall_curve(y_test, r['prob'])
    ax_pr.plot(rc, pc, lw=2.5, color=r['color'],
               label=f"{name} ({r['prauc']:.4f})")
ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
ax_pr.set_title('Precision-Recall Curves', fontweight='bold')
ax_pr.legend(labelcolor=C['white'], framealpha=0.3, fontsize=8)
ax_pr.grid(True, alpha=0.3)

# ── Row 2: Feature importance (best model) + bar comparison ───────────────
best_r  = final_results[best]
ax_fi   = fig.add_subplot(gs[2, 0:2])
if best == 'Random Forest':
    fi_use = fi_rf
elif best == 'XGBoost':
    fi_use = fi_xgb
else:
    fi_use = pd.Series(np.abs(lr_model.coef_[0]), index=FEATURES).sort_values(ascending=False)

fi_top = fi_use.head(12)
bc     = [C['red'] if v > fi_top.mean() else best_r['color'] for v in fi_top.values]
ax_fi.barh(fi_top.index[::-1], fi_top.values[::-1], color=bc[::-1], edgecolor='none', height=0.65)
ax_fi.set_title(f'Feature Importance — {best}', fontweight='bold')
ax_fi.set_xlabel('Score'); ax_fi.grid(True, axis='x', alpha=0.3)

# Bar chart metrics
ax_bar = fig.add_subplot(gs[2, 2:])
metric_keys2 = ['auc', 'prauc', 'f1', 'recall', 'precision']
x2 = np.arange(len(metric_keys2)); w2 = 0.26
for i2, (name, r) in enumerate(final_results.items()):
    vals2 = [r[k] for k in metric_keys2]
    ax_bar.bar(x2 + (i2-1)*w2, vals2, w2, label=name,
               color=r['color'], alpha=0.88, edgecolor='none')
ax_bar.set_xticks(x2)
ax_bar.set_xticklabels(['AUC','PR-AUC','F1','Recall','Precision'], fontsize=9)
ax_bar.set_ylim(0, 1.1); ax_bar.set_title('Final Metrics (tuned threshold)', fontweight='bold')
ax_bar.legend(labelcolor=C['white'], framealpha=0.3, fontsize=8)
ax_bar.grid(True, axis='y', alpha=0.3)

fig.suptitle('MASTER SUMMARY DASHBOARD — Credit Card Fraud Detection (SMOTE Training)',
             color=C['gold'], fontsize=15, fontweight='bold', y=1.01)
plt.savefig('reports/figures/ml_10_master_dashboard.png', dpi=130,
            bbox_inches='tight', facecolor=C['dark'])
plt.show()
print("✅  All 10 charts saved to reports/figures/")


## 11. Save Model & Artifacts <a id='s11'></a>

In [ ]:
# ── Save best model and all artifacts ─────────────────────────────────────
best_model_obj = final_results[best]['model']

joblib.dump(best_model_obj, 'ml/best_model.pkl')
joblib.dump(scaler,         'ml/scaler.pkl')

artifacts = {
    'best_model_name' : best,
    'features'        : FEATURES,
    'threshold'       : float(final_results[best]['thresh']),
    'roc_auc'         : float(final_results[best]['auc']),
    'pr_auc'          : float(final_results[best]['prauc']),
    'f1_fraud'        : float(final_results[best]['f1']),
    'recall_fraud'    : float(final_results[best]['recall']),
    'precision_fraud' : float(final_results[best]['precision']),
    'mcc'             : float(final_results[best]['mcc']),
    'cm'              : {
        'TN': int(final_results[best]['cm'][0,0]),
        'FP': int(final_results[best]['cm'][0,1]),
        'FN': int(final_results[best]['cm'][1,0]),
        'TP': int(final_results[best]['cm'][1,1]),
    },
    'smote_config'    : {
        'sampling_strategy' : 0.2,
        'k_neighbors'       : 5,
        'train_before_smote': [int((y_train==0).sum()), int(y_train.sum())],
        'train_after_smote' : [int((y_smote==0).sum()),  int(y_smote.sum())],
    },
    'all_models'      : {
        name: {
            'auc':       float(r['auc']),
            'prauc':     float(r['prauc']),
            'f1':        float(r['f1']),
            'recall':    float(r['recall']),
            'precision': float(r['precision']),
            'threshold': float(r['thresh']),
        }
        for name, r in final_results.items()
    },
    'methodology'     : (
        'Anti-leakage pipeline: '
        '(1) stratified 80/20 split first, '
        '(2) StandardScaler fit on X_train only, '
        '(3) SMOTE on training set only (strategy=0.2, k=5), '
        '(4) model trained on SMOTE dataset, '
        '(5) threshold tuned on train predictions, '
        '(6) test set evaluated once at the end.'
    )
}

with open('ml/final_metrics.json', 'w') as f:
    json.dump(artifacts, f, indent=2)

print("="*60)
print("  ARTIFACTS SAVED")
print("="*60)
print("  ml/best_model.pkl      ← trained model")
print("  ml/scaler.pkl          ← fitted scaler (train stats only)")
print("  ml/final_metrics.json  ← all metrics, config, methodology")
print()
print(f"  Best model   : {best}")
print(f"  ROC-AUC      : {final_results[best]['auc']:.4f}")
print(f"  PR-AUC       : {final_results[best]['prauc']:.4f}")
print(f"  Recall       : {final_results[best]['recall']:.4f}")
print(f"  Threshold    : {final_results[best]['thresh']:.2f}")
print(f"  Frauds caught: {final_results[best]['cm'][1,1]}/{y_test.sum()}")
print()
print("  Charts saved:")
for i in range(1,11):
    print(f"  reports/figures/ml_{i:02d}_*.png")
print("="*60)


In [ ]:
# ── How to use the saved model for new predictions ─────────────────────────
print("="*60)
print("  HOW TO USE SAVED MODEL FOR INFERENCE")
print("="*60)
print('''
import joblib, json, numpy as np, pandas as pd

# Load
model    = joblib.load('ml/best_model.pkl')
scaler   = joblib.load('ml/scaler.pkl')
meta     = json.load(open('ml/final_metrics.json'))
FEATURES = meta['features']
THRESHOLD= meta['threshold']

# Prepare a new transaction (must have same features)
new_txn = pd.DataFrame([{
    'V1': -1.36, 'V2': -0.07, 'V3': 2.54, 'V4': 1.38,
    'V5': -0.34, 'V6': 0.46,  'V7': 0.24, 'V8': 0.10,
    'V9': 0.36,  'V10': 0.09, 'V11': -0.55, 'V12': -0.62,
    'V13': -0.99,'V14': -0.31,'V15': 1.47,  'V16': -0.47,
    'V17': 0.21, 'V18': 0.03, 'V19': 0.40,  'V20': 0.25,
    'V21': -0.02,'V22': 0.28, 'V23': -0.11, 'V24': 0.07,
    'V25': 0.13, 'V26': -0.19,'V27': 0.13,  'V28': -0.02,
    'Amount': 149.62
}])

# Feature engineering (same as training)
new_txn['Amount_log']  = np.log1p(new_txn['Amount'])
new_txn['Amount_sqrt'] = np.sqrt(new_txn['Amount'])
new_txn['Hour']        = 0   # or derive from transaction timestamp

# Scale using the saved scaler (train statistics)
X_new = scaler.transform(new_txn[FEATURES])

# Predict
fraud_prob = model.predict_proba(X_new)[0, 1]
is_fraud   = fraud_prob >= THRESHOLD

print(f"Fraud probability : {fraud_prob:.4f}")
print(f"Decision          : {'🚨 FRAUD — BLOCK' if is_fraud else '✅ LEGITIMATE'}")
''')
